## Chapter 4: Getting to Know Your Data/Project: 
MultiLabel classification for Computer Vision /Ch4-01-Exploring Dataset.py

In [0]:
# from mlia_utils.cv_clf_funcs import *

# --- Inline definitions (replacing mlia_utils.cv_clf_funcs) ---
from PIL import Image
import matplotlib.pyplot as plt
from pyspark.sql import functions as f

volume_file_path = "/Volumes/porya_catalog/default/cv_clf"
base_path = f"{volume_file_path}/intel_image_clf/raw_images"
train_dir = f"{base_path}/seg_train/seg_train"
valid_dir = f"{base_path}/seg_test/seg_test"
train_delta_path = f"{volume_file_path}/train_imgs_main.delta"
valid_delta_path = f"{volume_file_path}/valid_imgs_main.delta"

def idx_class(df):
    rows = df.select("label_id", "label_name").distinct().collect()
    return {row["label_id"]: row["label_name"] for row in rows}

def display_image(path):
    img = Image.open(path)
    plt.figure()
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def proportion_labels(labels_dict):
    labels = list(labels_dict.keys())
    values = list(labels_dict.values())
    total = sum(values)
    proportions = [v / total for v in values]
    plt.figure(figsize=(10, 5))
    plt.bar(labels, proportions)
    plt.ylabel("Proportion")
    plt.title("Label Distribution")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def _count_labels(delta_path):
    df = spark.read.format("delta").load(delta_path)
    rows = df.groupBy("label_name").count().collect()
    return {row["label_name"]: row["count"] for row in rows}

labels_dict_train = _count_labels(train_delta_path)
labels_dict_valid = _count_labels(valid_delta_path)
# --- End inline definitions ---

train_df = (spark.read.format("delta").load(train_delta_path))
print(idx_class(train_df))
display(train_df)

# COMMAND ----------

display_image(f"{train_dir}/forest/17856.jpg")
display_image(f"{train_dir}/street/15478.jpg")

# COMMAND ----------

proportion_labels(labels_dict_train)

# COMMAND ----------

proportion_labels(labels_dict_valid)

In [0]:
import matplotlib.pyplot as plt

labels = list(labels_dict_train.keys())
counts = list(labels_dict_train.values())

plt.figure(figsize=(8, 8))
plt.pie(counts, labels=labels, autopct="%1.1f%%", startangle=140)
plt.title("Training Label Distribution")
plt.tight_layout()
plt.show()